In [4]:
import json

notebook = {
    "cells": [],
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "codemirror_mode": {"name": "ipython", "version": 3},
            "file_extension": ".py",
            "mimetype": "text/x-python",
            "name": "python",
            "nbconvert_exporter": "python",
            "pygments_lexer": "ipython3",
            "version": "3.8.0"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 5
}

def add_md(text):
    # Remove the trailing newline on the last element to keep it clean
    lines = [line + "\n" for line in text.split('\n')]
    if lines:
        lines[-1] = lines[-1].rstrip('\n')
    notebook["cells"].append({
        "cell_type": "markdown",
        "metadata": {},
        "source": lines
    })

def add_code(text):
    lines = [line + "\n" for line in text.split('\n')]
    if lines:
        lines[-1] = lines[-1].rstrip('\n')
    notebook["cells"].append({
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": lines
    })

# --- CELL 1 ---
add_md(r"""# Análisis Analítico de Amplificador Compuesto VFA-VFA (Modelo de 4 Polos)

Este cuaderno complementa el diseño analítico del amplificador compuesto, justificando los valores de los componentes de las redes de realimentación y demostrando el comportamiento exacto de los polos del sistema.

A diferencia de un modelo simplificado, aquí modelaremos la **función de transferencia completa de cuarto orden (4 polos)** resultante de anidar el lazo de A02 dentro del lazo global de A01. Esto nos permitirá comprobar de dónde surge el margen de fase de 62.5° observado en las simulaciones de LTSpice.""")

# --- CELL 2 ---
add_code("""import numpy as np
import control as ctrl
import matplotlib.pyplot as plt

# Parámetros nominales del VFA (LM324)
Ad0_dB = 100
Ad0 = 10**(Ad0_dB / 20)

f1 = 10.0          # Hz (Frecuencia del primer polo)
f2 = 5.06e6        # Hz (Frecuencia del segundo polo)

w1 = 2 * np.pi * f1
w2 = 2 * np.pi * f2

print(f"Ganancia Ad0: {Ad0:.1f} V/V")
print(f"Polo 1 (w1): {w1:.2f} rad/s")
print(f"Polo 2 (w2): {w2:.2f} rad/s")""")

# --- CELL 3 ---
add_md(r"""## 1. Diseño del Lazo Interno (A02) y Planicidad

El amplificador A02 posee una realimentación local $K_2 = R_1 / (R_1 + R_2)$. Al cerrar este lazo, sus dos polos de lazo abierto interactúan para formar un sistema de segundo orden. 

La ubicación de estos polos depende del factor de amortiguamiento $\zeta$. Para obtener la **máxima planicidad (Butterworth)**, desearíamos idealmente $\zeta = 0.707$. Sin embargo, con los parámetros del LM324, el amortiguamiento mínimo posible usando atenuación pasiva ($K_2 \le 1$) es superior a 1, por lo que el sistema quedará sobreamortiguado en su lazo interno.

Aquí puedes definir la ganancia a lazo cerrado que elegiste para A02 ($Av_{f2}$):""")

# --- CELL 4 ---
add_code("""# Definición de la ganancia objetivo para A02 (ajusta según tu diseño)
Avf2_target = 2.0  # Ejemplo: Ganancia de 2 V/V para A02
K2 = 1.0 / Avf2_target

# Cálculo de resistencias para lazo interno (fijando R1)
R1_A02 = 1000 # ohms
R2_A02 = R1_A02 * (Avf2_target - 1)

print(f"Red de realimentación A02 -> R1: {R1_A02} ohms, R2: {R2_A02} ohms")
print(f"Factor K2: {K2}")""")

# --- CELL 5 ---
add_md("""## 2. Modelado de las Funciones de Transferencia

Definimos el modelo de Laplace (`s`). Primero creamos la función de transferencia de lazo abierto $A_{d2}(s)$ y cerramos su lazo para obtener $A_{CL2}(s)$. Observaremos cómo los polos originales de A02 se han "desplazado" debido a su realimentación propia.""")

# --- CELL 6 ---
add_code("""s = ctrl.TransferFunction.s

# Planta de los amplificadores (iguales para A01 y A02)
A_open_loop = Ad0 / ((1 + s/w1) * (1 + s/w2))

# 1. Lazo Cerrado de A02
Acl2 = ctrl.feedback(A_open_loop, K2)

print("Polos originales de A02 (Lazo Abierto):")
print(ctrl.pole(A_open_loop) / (2*np.pi), "Hz\\n")

print("Nuevos polos de A02 (Lazo Cerrado):")
print(ctrl.pole(Acl2) / (2*np.pi), "Hz")""")

# --- CELL 7 ---
add_md(r"""## 3. Cierre del Lazo Global y Análisis de los 4 Polos

Al cerrar el lazo global, la planta que "ve" la red de realimentación externa es la cascada de $A01_{(lazo \\, abierto)} \times A02_{(lazo \\, cerrado)}$.
Esta planta combinada tiene **4 polos**. 

Al aplicar el factor de realimentación global $K_1$, **absolutamente todos los polos vuelven a moverse**. El polinomio característico ahora es de grado 4, y sus raíces nos darán la dinámica final del sistema.""")

# --- CELL 8 ---
add_code("""# Definición de la ganancia global del sistema compuesto
Avf1_target = 10.0 # Ajusta según la consigna de tu circuito
K1 = 1.0 / Avf1_target

# Lazo Directo (A01 en cascada con Acl2)
Lazo_Directo = A_open_loop * Acl2

# Lazo Cerrado Global
Acl_global = ctrl.feedback(Lazo_Directo, K1)

print("Polos finales del sistema (Lazo Cerrado Global):")
polos_finales = ctrl.pole(Acl_global)
for p in polos_finales:
    print(f"{p/(2*np.pi):.2e} Hz")""")

# --- CELL 9 ---
add_md(r"""## 4. Validación del Margen de Fase

Al evaluar el Margen de Fase usando el modelo exacto de cuarto orden, veremos que el retraso introducido por los polos "desplazados" de A02 (que a menudo se asumen a frecuencias muy altas y se desprecian en el cálculo a mano) sí afecta la fase en la frecuencia de cruce $\omega_c$. 

Esta es la justificación matemática precisa de por qué LTSpice reporta 62.5° en lugar de 65°.""")

# --- CELL 10 ---
add_code("""# Función de transferencia de Lazo Abierto Global (Lazo Directo * K1)
Lazo_Abierto_Global = Lazo_Directo * K1

# Cálculo exacto de márgenes
gm, pm, wg, wp = ctrl.margin(Lazo_Abierto_Global)

print(f"Frecuencia de cruce de ganancia (0 dB): {wp/(2*np.pi)/1000:.2f} kHz")
print(f"Margen de Fase Exacto: {pm:.2f} grados")

# Gráfico de Bode del lazo abierto global
plt.figure()
ctrl.bode_plot(Lazo_Abierto_Global, dB=True, margins=True)
plt.suptitle('Diagrama de Bode - Lazo Abierto Global (4 Polos)')
plt.show()""")

with open("Analisis_4_Polos_Amplificador_Compuesto.ipynb", "w", encoding="utf-8") as f:
    json.dump(notebook, f, indent=1, ensure_ascii=False)

print("[file-tag: code-generated-file-Analisis_4_Polos_Amplificador_Compuesto.ipynb]")

[file-tag: code-generated-file-Analisis_4_Polos_Amplificador_Compuesto.ipynb]
